# siyuan笔记集成 重构

---

本Notebook将思源笔记（SiYuan Note）与AI模型结合，实现自动化智能问答和财报处理功能。

## 1. 项目概述

### 1.1 功能描述

本项目主要功能包括：

1. **AI问答集成**：在思源笔记中调用通义千问长上下文模型（qwen-long）
2. **文档智能分析**：上传文档到AI模型并提问
3. **自动追加内容**：将AI回答自动追加到思源笔记块中
4. **财报文件OCR处理**：批量下载财报PDF并进行OCR识别

### 1.2 数据源

- **通义千问API**：提供AI问答服务（qwen-long模型）
- **思源笔记API**：笔记管理和内容追加
- **MySQL数据库**：yq.23年财报文件表
- **企业预警通API**：舆情数据获取

### 1.3 输出结果

- AI生成的回答内容（Markdown格式）
- 下载的PDF财报文件
- 数据库更新的OCR识别结果

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import json
import os
import re
import sys
import random
import shutil
from datetime import datetime, timedelta
from urllib import parse
import urllib.request
import urllib.parse

# 第三方库
import requests
import pandas as pd
import numpy as np
from sqlalchemy import text
import sqlalchemy
import sqlalchemy.pool

# 配置模块
from config import (
    DASHSCOPE_API_KEY,
    DASHSCOPE_API_URL,
    DB_USER,
    DB_PASSWORD,
    DB_HOST,
    DB_PORT,
    DB_NAME,
    SIYUAN_API_BASE,
    QYWYT_TOKEN,
    FOLDER_PATH
)

print("依赖导入成功！")

### 2.2 配置参数

In [ ]:
# 模型配置
MODEL_NAME = "qwen-long"

# 请求头配置
headers_dashscope = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {DASHSCOPE_API_KEY}"
}

headers_qywyt = {
    "Content-Type": "application/json",
    "Authorization": f"token {QYWYT_TOKEN}"
}

# 思源笔记配置
SIYUAN_NOTEBOOK_ID = "20240711131333-a9cy8wy"  # 笔记本ID

print("配置参数加载成功！")

## 3. 数据获取

### 3.1 通义千问API连接

In [ ]:
def ask_model(question, file_id=None):
    """
    调用通义千问API进行问答
    
    参数:
        question: 用户提问内容
        file_id: 文档ID（可选），用于上传文档后提问
    
    返回:
        AI模型的回答内容
    """
    # 构建系统消息
    system_messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]
    
    # 如果有文档ID，添加到系统消息
    if file_id:
        system_messages.append({"role": "system", "content": f"fileid://{file_id}"})
    
    # 构建请求数据
    payload = {
        "model": MODEL_NAME,
        "messages": [
            *system_messages,
            {"role": "user", "content": question}
        ],
        "stream": False
    }
    
    try:
        response = requests.post(
            DASHSCOPE_API_URL,
            headers=headers_dashscope,
            json=payload,
            timeout=60
        )
        
        if response.status_code != 200:
            raise Exception(f"API请求失败，状态码: {response.status_code}")
        
        result = response.json()
        
        if "choices" in result and len(result["choices"]) > 0:
            return result["choices"][0]["message"]["content"]
        else:
            raise Exception("响应结构异常")
            
    except requests.exceptions.RequestException as e:
        raise Exception(f"网络请求错误: {e}")
    except json.JSONDecodeError as e:
        raise Exception(f"JSON解析错误: {e}")


# 测试API连接
def test_dashscope_connection():
    """测试通义千问API连接"""
    try:
        result = ask_model("你好，请回复'连接成功'")
        print(f"API连接测试成功！响应: {result[:50]}...")
        return True
    except Exception as e:
        print(f"API连接测试失败: {e}")
        return False

# 执行测试（需要有效API Key）
# test_dashscope_connection()

### 3.2 思源笔记API连接

In [ ]:
def get_siyuan_notebooks():
    """
    获取思源笔记笔记本列表
    
    返回:
        笔记本列表
    """
    url = f"{SIYUAN_API_BASE}/api/notebook/lsNotebooks"
    response = requests.post(url, headers={"Content-Type": "application/json"})
    return response.json()


def create_doc_with_md(notebook_id, path, markdown):
    """
    使用Markdown创建文档
    
    参数:
        notebook_id: 笔记本ID
        path: 文档路径
        markdown: Markdown内容
    
    返回:
        API响应
    """
    url = f"{SIYUAN_API_BASE}/api/filetree/createDocWithMd"
    data = {
        "notebook": notebook_id,
        "path": path,
        "markdown": markdown
    }
    response = requests.post(url, headers={"Content-Type": "application/json"}, json=data)
    return response


def append_block(data_type, data, parent_id):
    """
    追加内容块到思源笔记
    
    参数:
        data_type: 数据类型（如"markdown"）
        data: 内容数据
        parent_id: 父块ID
    
    返回:
        API响应
    """
    url = f"{SIYUAN_API_BASE}/api/block/appendBlock"
    payload = {
        "dataType": data_type,
        "data": data,
        "parentID": parent_id
    }
    response = requests.post(url, headers={"Content-Type": "application/json"}, json=payload)
    return response


# 测试思源笔记连接
def test_siyuan_connection():
    """测试思源笔记API连接"""
    try:
        result = get_siyuan_notebooks()
        if result.get("code") == 0:
            print(f"思源笔记连接成功！笔记本数量: {len(result.get('data', []))}")
            return True
        else:
            print(f"思源笔记连接失败: {result.get('msg')}")
            return False
    except Exception as e:
        print(f"思源笔记连接测试失败: {e}")
        return False

# 执行测试（需要思源笔记运行中）
# test_siyuan_connection()

### 3.3 数据库连接

In [ ]:
def create_db_engine():
    """
    创建数据库连接引擎
    
    返回:
        SQLAlchemy引擎对象
    """
    connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = sqlalchemy.create_engine(
        connection_string,
        poolclass=sqlalchemy.pool.NullPool
    )
    return engine


# 创建数据库引擎
sql_engine = create_db_engine()
print("数据库引擎创建成功！")


def query_financial_reports():
    """
    查询23年财报文件
    
    返回:
        包含财报信息的DataFrame
    """
    sql = """
    SELECT A.trade_code, A.fileName, A.公司名称 as company
    FROM yq.23年财报文件 A
    WHERE A.fileName != ''
    """
    
    with sql_engine.begin() as connection:
        df = pd.read_sql(sql, connection)
    
    return df


# 测试数据库连接
def test_db_connection():
    """测试数据库连接"""
    try:
        df = query_financial_reports()
        print(f"数据库连接成功！查询到 {len(df)} 条财报记录")
        return True
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return False

# 执行测试
# test_db_connection()

## 4. 数据处理

### 4.1 文件名清理

In [ ]:
def sanitize_filename(filename):
    """
    清理文件名，使其符合Windows文件命名规范
    
    参数:
        filename: 原始文件名
    
    返回:
        清理后的文件名
    """
    # Windows文件名禁止字符: < > : " / \ | ? *
    sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
    return sanitized


# 测试文件名清理
test_names = [
    "2024年财报:第一季度",
    "报告<测试>.pdf",
    "数据/分析|结果"
]

for name in test_names:
    print(f"原始: {name} -> 清理后: {sanitize_filename(name)}")

### 4.2 PDF下载

In [ ]:
def download_pdf(file_url, file_name, folder_path=None):
    """
    下载PDF文件
    
    参数:
        file_url: 文件URL
        file_name: 文件名（不含扩展名）
        folder_path: 保存目录（默认使用配置中的路径）
    
    返回:
        文件保存路径
    """
    if folder_path is None:
        folder_path = FOLDER_PATH
    
    # 确保目录存在
    os.makedirs(folder_path, exist_ok=True)
    
    # 清理文件名
    safe_name = sanitize_filename(file_name)
    file_path = os.path.join(folder_path, f"{safe_name}.pdf")
    
    # 检查文件是否已存在
    if os.path.exists(file_path):
        print(f"文件已存在: {safe_name}.pdf")
        return file_path
    
    try:
        response = requests.get(file_url, timeout=30)
        response.raise_for_status()
        
        with open(file_path, "wb") as f:
            f.write(response.content)
        
        print(f"下载成功: {safe_name}.pdf")
        return file_path
        
    except requests.exceptions.RequestException as e:
        print(f"下载失败: {safe_name}.pdf - {e}")
        return None


def batch_download_pdfs(df, url_column, name_column):
    """
    批量下载PDF文件
    
    参数:
        df: 包含URL和文件名的DataFrame
        url_column: URL列名
        name_column: 文件名列名
    
    返回:
        下载结果列表
    """
    results = []
    
    for idx, row in df.iterrows():
        file_url = row[url_column]
        file_name = row[name_column]
        
        if pd.notna(file_url) and pd.notna(file_name):
            result = download_pdf(file_url, file_name)
            results.append({
                "index": idx,
                "file_name": file_name,
                "status": "success" if result else "failed"
            })
    
    return results

## 5. 核心逻辑

### 5.1 AI问答功能

In [ ]:
def ask_and_append_to_siyuan(question, parent_id, file_id=None):
    """
    AI问答并追加到思源笔记
    
    参数:
        question: 提问内容
        parent_id: 父块ID
        file_id: 文档ID（可选）
    
    返回:
        AI回答内容
    """
    try:
        # 获取AI回答
        model_answer = ask_model(question, file_id)
        
        # 构建Markdown格式的内容
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        markdown_content = f"""## AI回答

{model_answer}

---
*生成时间: {timestamp}*
"""
        
        # 追加到思源笔记
        response = append_block("markdown", markdown_content, parent_id)
        
        if response.status_code == 200:
            print("内容已成功追加到思源笔记")
        else:
            print(f"追加失败: {response.text}")
        
        return model_answer
        
    except Exception as e:
        print(f"AI问答失败: {e}")
        return None


def ask_with_retry(question, file_id=None, max_retries=3):
    """
    带重试机制的AI问答
    
    参数:
        question: 提问内容
        file_id: 文档ID（可选）
        max_retries: 最大重试次数
    
    返回:
        AI回答内容
    """
    last_error = None
    
    for i in range(max_retries):
        try:
            return ask_model(question, file_id)
        except Exception as e:
            last_error = e
            print(f"第 {i + 1} 次尝试失败: {e}")
            if i < max_retries - 1:
                import time
                time.sleep(2 ** i)  # 指数退避
    
    raise last_error

### 5.2 文档上传与提问

In [ ]:
def ask_about_document(question, file_id):
    """
    针对文档提问
    
    参数:
        question: 提问内容
        file_id: 已上传的文档ID
    
    返回:
        AI回答内容
    """
    # 文档引用格式: fileid://{fileId}
    return ask_model(question, file_id=file_id)


# 示例：使用文档ID提问
def document_qa_example():
    """文档问答示例"""
    question = "这份报告的主要结论是什么？"
    file_id = "file-fe-1LuMGonfgr3qgC48dm40uBCU"  # 示例文档ID
    
    try:
        answer = ask_about_document(question, file_id)
        print(f"问题: {question}")
        print(f"回答: {answer}")
        return answer
    except Exception as e:
        print(f"文档问答失败: {e}")
        return None

# document_qa_example()

### 5.3 财报OCR处理

In [ ]:
def update_ocr_status(origin_filename, ocr_result):
    """
    更新财报文件的OCR识别结果
    
    参数:
        origin_filename: 原始文件名
        ocr_result: OCR识别结果
    """
    sql = """
    UPDATE 23年财报文件
    SET ocr = :ocr_result
    WHERE fileName = :origin_filename
    """
    
    params = {
        "origin_filename": origin_filename,
        "ocr_result": ocr_result
    }
    
    with sql_engine.begin() as connection:
        connection.execute(text(sql), params)
    
    print(f"已更新OCR结果: {origin_filename}")


def update_file_url(trade_code, file_url, file_name):
    """
    更新财报文件的URL和文件名
    
    参数:
        trade_code: 债券代码
        file_url: 文件URL
        file_name: 文件名
    """
    sql = """
    UPDATE 23年财报文件
    SET fileUrl = :fileUrl, fileName = :fileName, ocr = '已重下'
    WHERE trade_code = :trade_code
    """
    
    params = {
        "trade_code": trade_code,
        "fileUrl": file_url,
        "fileName": file_name
    }
    
    with sql_engine.begin() as connection:
        connection.execute(text(sql), params)
    
    print(f"已更新文件信息: {trade_code}")

### 5.4 企业预警通舆情处理

In [ ]:
def get_public_opinion_detail(opinion_id):
    """
    获取企业预警通舆情详情
    
    参数:
        opinion_id: 舆情ID
    
    返回:
        舆情详情字典
    """
    url = f"https://host.ratingdog.cn/api/services/app/ResearchInfo/GetCiPublicOpinionForView?id={opinion_id}"
    
    try:
        response = requests.get(url, headers=headers_qywyt, json={"id": opinion_id})
        response.raise_for_status()
        result = response.json()
        
        if result.get('result') is None:
            return None
        
        return result['result']
        
    except Exception as e:
        print(f"获取舆情详情失败: {e}")
        return None


def save_sentiment_to_db(list_keywords, type_info, happen_date, content_info, ai_sum):
    """
    保存舆情信息到数据库
    
    参数:
        list_keywords: 涉及公司
        type_info: 信息类型
        happen_date: 发生日期
        content_info: 内容信息
        ai_sum: AI摘要
    """
    sql = """
    INSERT IGNORE INTO 金融债舆情监控 
    (list_keywords, TypeInfo, happenDate, ContentInfo, ai_sum)
    VALUES (:list_keywords, :TypeInfo, :happenDate, :ContentInfo, :ai_sum)
    """
    
    params = {
        "list_keywords": list_keywords,
        "TypeInfo": type_info,
        "happenDate": happen_date,
        "ContentInfo": content_info,
        "ai_sum": ai_sum
    }
    
    with sql_engine.begin() as connection:
        connection.execute(text(sql), params)
    
    print(f"已保存舆情信息: {list_keywords[:50]}...")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def main_workflow():
    """
    主工作流程
    """
    print("=" * 50)
    print("siyuan笔记集成 - 主工作流程")
    print("=" * 50)
    
    # 1. 测试连接
    print("\n1. 测试各服务连接...")
    connections = {
        "思源笔记": test_siyuan_connection(),
        "数据库": test_db_connection()
    }
    
    for name, status in connections.items():
        print(f"   {name}: {'成功' if status else '失败'}")
    
    # 2. 查询财报数据
    print("\n2. 查询财报数据...")
    try:
        df_reports = query_financial_reports()
        print(f"   查询到 {len(df_reports)} 条财报记录")
        if len(df_reports) > 0:
            print(f"   示例: {df_reports.head(3).to_string()}")
    except Exception as e:
        print(f"   查询失败: {e}")
    
    print("\n" + "=" * 50)
    print("工作流程完成！")
    print("=" * 50)


# 执行主流程
if __name__ == "__main__":
    main_workflow()

### 6.2 结果验证

In [ ]:
def verify_environment():
    """
    验证环境配置
    """
    print("环境验证结果:")
    print("-" * 30)
    
    # 检查API密钥
    api_key_status = "已配置" if DASHSCOPE_API_KEY and DASHSCOPE_API_KEY != "your_api_key_here" else "未配置"
    print(f"通义千问API Key: {api_key_status}")
    
    # 检查数据库配置
    db_status = "已配置" if DB_HOST and DB_USER else "未配置"
    print(f"数据库配置: {db_status}")
    
    # 检查目录
    folder_status = "存在" if os.path.exists(FOLDER_PATH) else "不存在"
    print(f"下载目录: {folder_status}")
    
    print("-" * 30)


verify_environment()

## 7. 工具函数（可复用）

以下函数可在其他项目中复用：

In [ ]:
# ==================== 工具函数集合 ====================

def safe_request(url, method="GET", headers=None, json_data=None, timeout=30, max_retries=3):
    """
    安全的HTTP请求函数（带重试机制）
    
    参数:
        url: 请求URL
        method: 请求方法
        headers: 请求头
        json_data: JSON数据
        timeout: 超时时间
        max_retries: 最大重试次数
    
    返回:
        响应对象
    """
    import time
    
    for attempt in range(max_retries):
        try:
            if method.upper() == "GET":
                response = requests.get(url, headers=headers, json=json_data, timeout=timeout)
            else:
                response = requests.post(url, headers=headers, json=json_data, timeout=timeout)
            
            response.raise_for_status()
            return response
            
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"请求失败，{wait_time}秒后重试... ({attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                raise e


def format_timestamp(dt=None):
    """
    格式化时间戳
    
    参数:
        dt: datetime对象（默认为当前时间）
    
    返回:
        格式化的时间字符串
    """
    if dt is None:
        dt = datetime.now()
    return dt.strftime("%Y-%m-%d %H:%M:%S")


def ensure_directory(path):
    """
    确保目录存在
    
    参数:
        path: 目录路径
    """
    os.makedirs(path, exist_ok=True)
    return path


def truncate_text(text, max_length=100, suffix="..."):
    """
    截断文本
    
    参数:
        text: 原始文本
        max_length: 最大长度
        suffix: 后缀
    
    返回:
        截断后的文本
    """
    if len(text) <= max_length:
        return text
    return text[:max_length - len(suffix)] + suffix


print("工具函数加载完成！")
print("可用函数: safe_request, format_timestamp, ensure_directory, truncate_text")